# Stage 8: turning the forecast into "Low / Medium / High"

The models output a number: *"expected daily move 1.8%"*. The app shows a **risk level**. Two ways to do that:

1. **Compare the stock with itself** ("is this a wiggly week *for this stock*?"). Problem: a very jumpy stock would often be called "Low" and a very calm one "High", which would mislead people comparing them.
2. **Compare the stock with all stocks** ("is this a wiggly week *compared with the market*?"). Low means "calmer than most Indian stocks". This is easier to understand and compare.

We use **(2) for the risk level**, and show **(1) as a second small label**: *"Calmer than usual / About usual / Wigglier than usual"*, so you also see whether this week is a change for the stock.

**The cut-offs.** We take the models' **honest out-of-sample guesses** (the ensemble of GARCH, XGBoost and LSTM from the scoreboard) for the **most recent 12 months**, across all stocks, and cut them into thirds: the calmest third is **Low**, the wiggliest third is **High**, the rest **Medium**. The two cut-off numbers are then fixed, so a label means the same thing every night. We re-check them about once a year.

Kernel: **RiskWise (.venv)**.

In [ ]:
# Cell 1: load the models' exam-period guesses (from the scoreboard notebook)
import sys
sys.path.append("../pipeline")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from risk import level, vs_usual

t = pd.read_parquet("../data/exam_table.parquet")
t["date"] = pd.to_datetime(t["date"])
last_year = t[t["date"] > t["date"].max() - pd.DateOffset(years=1)]
print(f"{len(t):,} exam rows; last 12 months: {len(last_year):,} rows, {last_year['sym'].nunique()} stocks")

In [ ]:
# Cell 2: what do the ensemble's guesses look like across the market?
print("expected daily move (% a day), last 12 months, all stocks:")
print(last_year["ens_all"].describe(percentiles=[.1, .33, .5, .67, .9]).round(2).to_string())
last_year["ens_all"].plot.hist(bins=80, figsize=(8, 3), title="Spread of expected daily moves (% a day)")
plt.xlim(0, 8)
plt.show()

## Choosing the two cut-offs

In [ ]:
# Cell 3: the cut-offs are the 1/3 and 2/3 points of the last year's guesses
low_below, high_from = np.quantile(last_year["ens_all"], [1 / 3, 2 / 3])
bands = {"low_below": round(float(low_below), 2), "high_from": round(float(high_from), 2),
         "made_from": "ensemble out-of-sample guesses, last 12 months to " + str(t["date"].max().date())}
print(bands)
print(f"\nLow: expected daily move below {bands['low_below']}%   |   Medium: between   |   High: {bands['high_from']}% or more")

## Do the labels mean something?

If "High" really means "wigglier week ahead", then the weeks labelled High should turn out wigglier than the weeks labelled Low. We check with the **real** wiggle that followed (never used for the label).

In [ ]:
# Cell 4: label every exam row, then compare with what really happened
t["level"] = t["ens_all"].apply(lambda w: level(w, bands))
check = t.groupby("level")["actual_next5"].describe(percentiles=[.5, .9])[["count", "50%", "90%"]].reindex(["Low", "Medium", "High"]).round(2)
check.columns = ["rows", "typical real wiggle (% a day)", "wiggliest 10% of real weeks"]
display(check)

low_rows = t[t["level"] == "Low"]
high_rows = t[t["level"] == "High"]
print(f"\nWeeks labelled Low that turned out wigglier than the High cut-off ({bands['high_from']}%): {(low_rows['actual_next5'] >= bands['high_from']).mean() * 100:.1f}%")
print(f"Weeks labelled High that turned out calmer than the Low cut-off ({bands['low_below']}%): {(high_rows['actual_next5'] < bands['low_below']).mean() * 100:.1f}%")

Those last two numbers are the **honest error rates of the label**. They are worth showing on the How it works page: a "Low" is a forecast, not a promise.

## Is the label steady over time, or does it flip all the time?

If a stock's label flipped every other day, the app would feel jumpy and untrustworthy. We take each stock's label once a week and count how often it changes from one week to the next.

In [ ]:
# Cell 5: week-to-week label changes, and how the labels split in each exam block
ordered = t.sort_values(["sym", "date"])
weekly = ordered[ordered.groupby("sym").cumcount() % 5 == 0].copy()          # one row per stock per 5 trading days
weekly["prev"] = weekly.groupby("sym")["level"].shift(1)
changed = (weekly["level"] != weekly["prev"])[weekly["prev"].notna()].mean() * 100
print(f"share of weekly labels that differ from the previous week's: {changed:.1f}%")

split = t.groupby("block")["level"].value_counts(normalize=True).unstack().reindex(columns=["Low", "Medium", "High"]) * 100
print("\nshare of Low / Medium / High in each 6-month block of the exam (%), using the SAME fixed cut-offs:")
display(split.round(1))

If the three shares stay roughly one third each over the blocks, the fixed cut-offs still make sense. If they drift a lot (for example, a market-wide calm spell makes almost everything "Low"), we need to re-check the cut-offs more often.

## The second label: "vs its usual"

The stock's own **usual** is the median of **the model's own forecasts over the previous 250 days** (using only past data), so we compare like with like. Forecast below 85% of usual is *calmer than usual*, above 115% is *wigglier than usual*. (Our first try compared forecasts with the median of past *real* wiggles, which are not the same kind of number, and it labelled almost a third of all weeks 'wigglier than usual'.)

In [ ]:
# Cell 6: "vs its usual" label
ordered = t.sort_values(["sym", "date"]).copy()
ordered["usual"] = ordered.groupby("sym")["ens_all"].transform(lambda s: s.rolling(250, min_periods=120).median().shift(1))   # past forecasts only
t2 = ordered.dropna(subset=["usual"]).copy()
t2["vs_usual"] = [vs_usual(w, u) for w, u in zip(t2["ens_all"], t2["usual"])]
print((t2["vs_usual"].value_counts(normalize=True) * 100).round(1).to_string())

In [ ]:
# Cell 7: save the cut-offs for the nightly job
Path("../models").mkdir(exist_ok=True)
Path("../models/risk_bands.json").write_text(json.dumps(bands, indent=2))
print("saved ../models/risk_bands.json:", bands)

## Reading the results honestly

- The label is **relative to today's market**: "Low" means calmer than about two-thirds of the market, not "safe". We must keep the words "Low risk" paired with "forecast, can be wrong" on the website.
- Check the two error rates in Cell 4. They tell you how often a Low week turned stormy and a High week turned calm.
- The cut-offs come from one recent year. In a very calm or very panicky year they would shift, which is why we re-check them yearly.

## Your turn

1. In Cell 3, change the quantiles from thirds to `[0.4, 0.8]` (40% Low, 40% Medium, 20% High). Which is easier to explain to a user?
2. In Cell 5, how often does a stock's label change week to week? Would you add a rule to make labels steadier?
3. Explain back: **why do we use the models' out-of-sample guesses, not their in-sample guesses, to choose the cut-offs?**